In [2]:
"""
AgroAlert Ghana -- Extended Feature Engineering (2019-2023)
=============================================================
Identical logic to original 05_data_processing.ipynb (final cell).
Reads the three EXTENDED raw files and produces an expanded master_dataset.

No changes to feature engineering logic itself -- this preserves comparability
with your original 2022-2023 results (same rolling windows, same SPEI proxy
formula, same drought-label thresholds), so reviewers can see the extension
is a like-for-like scale-up, not a re-designed pipeline.
"""

import pandas as pd
import numpy as np
import os

RAW_DIR = 'C:/Users/ELITE/Documents/AGROALERT/data_raw'
OUT_DIR = 'C:/Users/ELITE/Documents/AGROALERT/data_processed'

# ---- Load extended raw datasets ----
df_ndvi = pd.read_csv(f'{RAW_DIR}/ndvi_raw_2019_2023.csv')
df_weather = pd.read_csv(f'{RAW_DIR}/weather_raw_2019_2023.csv')
df_lst = pd.read_csv(f'{RAW_DIR}/soil_moisture_raw_2019_2023.csv')

df_ndvi['date'] = pd.to_datetime(df_ndvi['date'])
df_weather['date'] = pd.to_datetime(df_weather['date'])
df_lst['date'] = pd.to_datetime(df_lst['date'])

print("NDVI shape:", df_ndvi.shape)
print("Weather shape:", df_weather.shape)
print("LST shape:", df_lst.shape)

# ---- Resample to weekly (identical logic to original) ----
df_ndvi_w = (df_ndvi.groupby(['community', 'region',
             pd.Grouper(key='date', freq='W')])['ndvi']
             .mean().reset_index())

df_weather_w = (df_weather.groupby(['community', 'region',
               pd.Grouper(key='date', freq='W')])
               .agg({'rainfall_mm': 'sum', 'temp_max': 'mean',
                     'temp_min': 'mean', 'humidity': 'mean', 'et0': 'sum'})
               .reset_index())

df_lst_w = (df_lst.groupby(['community', 'region',
            pd.Grouper(key='date', freq='W')])['lst_celsius']
            .mean().reset_index())

# ---- Merge ----
df = df_ndvi_w.merge(df_weather_w, on=['community', 'region', 'date'], how='outer')
df = df.merge(df_lst_w, on=['community', 'region', 'date'], how='outer')
df = df.sort_values(['community', 'date']).reset_index(drop=True)

print(f"\nMerged shape (before gap-filling): {df.shape}")
print(f"Missing values before filling:\n{df.isnull().sum()}")

# ---- Gap-filling (identical to original: linear interpolation, then bfill/ffill) ----
for community in df['community'].unique():
    mask = df['community'] == community
    df.loc[mask, 'ndvi'] = df.loc[mask, 'ndvi'].interpolate().bfill().ffill()
    df.loc[mask, 'lst_celsius'] = df.loc[mask, 'lst_celsius'].interpolate().bfill().ffill()

# ---- Feature engineering (identical formulas to original) ----
df['ndvi_anomaly'] = (df.groupby('community')['ndvi']
                      .transform(lambda x: x - x.rolling(4, min_periods=1).mean()))
df['rainfall_deficit'] = (df.groupby('community')['rainfall_mm']
                          .transform(lambda x: x - x.rolling(4, min_periods=1).mean()))
df['water_balance'] = df['rainfall_mm'] - df['et0']
df['spei_proxy'] = (df.groupby('community')['water_balance']
                    .transform(lambda x:
                        (x.rolling(4, min_periods=1).sum() -
                         x.rolling(4, min_periods=1).sum().mean()) /
                        (x.rolling(4, min_periods=1).sum().std() + 1e-8)))

# ---- Drought label (identical dual-condition threshold) ----
df['drought_label'] = (
    (df['spei_proxy'] < -1.0) &
    (df['ndvi_anomaly'] < -0.05)
).astype(int)

# ---- Save ----
os.makedirs(OUT_DIR, exist_ok=True)
OUTPUT_PATH = f'{OUT_DIR}/master_dataset_2019_2023.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f"\nFinal dataset shape: {df.shape}")
print(f"Saved to: {OUTPUT_PATH}")
print(f"Communities: {df['community'].nunique()}")
print(f"Missing values remaining: {df.isnull().sum().sum()}")
print(f"\nDrought label distribution:")
print(df['drought_label'].value_counts())
print(f"\nDrought cases per community:")
print(df.groupby('community')['drought_label'].sum().sort_values(ascending=False))
print(f"\nDrought cases per year:")
print(df.groupby(df['date'].dt.year)['drought_label'].sum())

print("\n--- COMPARISON TO ORIGINAL 2022-2023 DATASET ---")
print("Original: 1,575 rows, 29 drought weeks (1.8% prevalence)")
print(f"Extended: {len(df)} rows, {df['drought_label'].sum()} drought weeks "
      f"({100*df['drought_label'].sum()/len(df):.1f}% prevalence)")


NDVI shape: (2047, 4)
Weather shape: (27390, 8)
LST shape: (2933, 4)

Merged shape (before gap-filling): (3915, 10)
Missing values before filling:
community         0
region            0
date              0
ndvi           2572
rainfall_mm       0
temp_max          0
temp_min          0
humidity          0
et0               0
lst_celsius    1012
dtype: int64

Final dataset shape: (3915, 15)
Saved to: C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset_2019_2023.csv
Communities: 15
Missing values remaining: 0

Drought label distribution:
drought_label
0    3858
1      57
Name: count, dtype: int64

Drought cases per community:
community
Koforidua           11
Ho                  11
Goaso                9
Sunyani              7
Techiman             7
Sefwi Wiawso         3
Bolgatanga           2
Cape Coast           2
Kumasi               2
Sekondi-Takoradi     1
Dambai               1
Nalerigu             1
Damongo              0
Tamale               0
Wa                   0


In [3]:
import pandas as pd
df = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset_2019_2023.csv')

print("Drought weeks per year:")
print(df.groupby(pd.to_datetime(df['date']).dt.year)['drought_label'].sum())

print("\nDrought weeks per community:")
print(df.groupby('community')['drought_label'].sum().sort_values(ascending=False))

print("\nRows per community (check for consistency):")
print(df['community'].value_counts().sort_index())

Drought weeks per year:
date
2019     2
2020    16
2021    14
2022     9
2023    16
Name: drought_label, dtype: int64

Drought weeks per community:
community
Koforidua           11
Ho                  11
Goaso                9
Sunyani              7
Techiman             7
Sefwi Wiawso         3
Bolgatanga           2
Cape Coast           2
Kumasi               2
Sekondi-Takoradi     1
Dambai               1
Nalerigu             1
Damongo              0
Tamale               0
Wa                   0
Name: drought_label, dtype: int64

Rows per community (check for consistency):
community
Bolgatanga          261
Cape Coast          261
Dambai              261
Damongo             261
Goaso               261
Ho                  261
Koforidua           261
Kumasi              261
Nalerigu            261
Sefwi Wiawso        261
Sekondi-Takoradi    261
Sunyani             261
Tamale              261
Techiman            261
Wa                  261
Name: count, dtype: int64


In [4]:
# Check raw NDVI coverage before gap-filling, per community per year
df_ndvi_check = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_raw/ndvi_raw_2019_2023.csv')
df_ndvi_check['year'] = pd.to_datetime(df_ndvi_check['date']).dt.year
print(df_ndvi_check.groupby(['community', 'year']).size().unstack(fill_value=0))


year              2019  2020  2021  2022  2023
community                                     
Bolgatanga          65    62    70    46    51
Cape Coast           8    14    11     4     7
Dambai              66    70    85    52    65
Damongo             27    29    26    21    26
Goaso               25    30    17    20    17
Ho                  25    45    37    30    27
Koforidua           35    40    34    23    25
Kumasi              18    14     9    10     8
Nalerigu            29    31    32    22    25
Sefwi Wiawso         9     6     4     4     3
Sekondi-Takoradi     8    14    11     4     7
Sunyani             38    39    24    24    33
Tamale              28    27    29    19    22
Techiman            20    24    24    16    18
Wa                  30    40    37    29    23
